# Retail Business Analysis

This notebook calculates the main sales, order, product, country, customer, and cancellation KPIs from the cleaned retail datasets.

Only valid completed transactions are used for gross sales analysis.The complete cleaned transaction dataset is used when analyzing cancellations and negative-quantity activity.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

processed_data_dir = (project_root/ "data"/ "processed")

reports_table_dir = (project_root/ "reports"/ "tables")

reports_table_dir.mkdir(parents=True,exist_ok=True,)

clean_transactions_path = (processed_data_dir/ "clean_transactions.csv")

valid_sales_path = (processed_data_dir/ "valid_sales.csv")

customer_sales_path = (processed_data_dir/ "customer_sales.csv")

print(f"Project root: {project_root}")
print("Clean transactions available:",
    clean_transactions_path.exists(),
)
print("Valid sales available:",
    valid_sales_path.exists(),
)
print("Customer sales available:",
    customer_sales_path.exists(),
)

Project root: c:\Users\ASUS\Python_workspace\retail_sales_insights
Clean transactions available: True
Valid sales available: True
Customer sales available: True


## 1. Load Processed Data

In [3]:
transactions_df = pd.read_csv(clean_transactions_path,parse_dates=["invoice_date","invoice_month"],)

sales_df = pd.read_csv(valid_sales_path,parse_dates=["invoice_date","invoice_month",])

customer_sales_df = pd.read_csv(customer_sales_path,parse_dates=["invoice_date","invoice_month",])

C:\Users\ASUS\AppData\Local\Temp\ipykernel_14380\1988774456.py:3: DtypeWarning: Columns (0: invoice_no) have mixed types. Specify dtype option on import or set low_memory=False.
  sales_df = pd.read_csv(valid_sales_path,parse_dates=["invoice_date","invoice_month",])


In [4]:
transactions_df["customer_id"] = (transactions_df["customer_id"].astype("Int64"))

sales_df["customer_id"] = (sales_df["customer_id"].astype("Int64"))

customer_sales_df["customer_id"] = (customer_sales_df["customer_id"].astype("Int64"))

In [5]:
dataframes = [transactions_df, sales_df, customer_sales_df]

for dataframe in dataframes:
    dataframe["invoice_no"] = (dataframe["invoice_no"].astype("string"))

    dataframe["stock_code"] = (dataframe["stock_code"].astype("string"))

In [6]:
print("Clean transactions:", transactions_df.shape,)

print("Valid sales:", sales_df.shape,)

print("Customer sales:", customer_sales_df.shape,)

Clean transactions: (535187, 17)
Valid sales: (524878, 17)
Customer sales: (392692, 17)


## 2. Validate the Input Data

In [7]:
assert not transactions_df.empty
assert not sales_df.empty
assert not customer_sales_df.empty

assert (sales_df["quantity"] > 0).all()
assert (sales_df["unit_price"] > 0).all()
assert (sales_df["line_revenue"] > 0).all()
assert (~sales_df["is_cancelled"]).all()

assert (customer_sales_df["customer_id"].notna().all())

print("Input datasets are valid.")

Input datasets are valid.


## 3. KPI Definitions

The following definitions are used:

- **Gross sales revenue:** Sum of line revenue from valid completed sales.
- **Completed orders:** Number of unique non-cancelled invoice numbers.
- **Units sold:** Sum of quantity from valid completed sales.
- **Unique customers:** Number of known customer IDs in completed sales.
- **Average order value:** Gross revenue divided by completed orders.
- **Average items per order:** Units sold divided by completed orders.
- **Cancellation rate:** Unique cancelled invoice numbers divided by all unique
  invoice numbers.
- **Negative-quantity value:** Absolute value of negative transaction revenue.

Rows with missing customer IDs are included in general sales KPIs but excluded from customer-level KPIs.

## 4. Overall Business KPIs

In [8]:
gross_revenue = (sales_df["line_revenue"].sum())

completed_orders = (sales_df["invoice_no"].nunique())

units_sold = (sales_df["quantity"].sum())

unique_customers = (customer_sales_df["customer_id"].nunique())

average_order_value = (gross_revenue/ completed_orders)

average_items_per_order = (units_sold/ completed_orders)

known_customer_revenue = (customer_sales_df["line_revenue"].sum())

average_revenue_per_customer = (known_customer_revenue/ unique_customers)

In [9]:
all_invoice_count = (transactions_df["invoice_no"].nunique())

cancelled_invoice_count = (
        transactions_df.loc[
        transactions_df["is_cancelled"],
        "invoice_no",
    ].nunique()
)

cancellation_rate = (cancelled_invoice_count/ all_invoice_count* 100)

In [10]:
negative_quantity_mask = ((transactions_df["quantity"] < 0)& (transactions_df["unit_price"] > 0))

negative_quantity_rows = (transactions_df[negative_quantity_mask])

returned_or_cancelled_units = (-negative_quantity_rows["quantity"].sum())

negative_quantity_value = (-negative_quantity_rows["line_revenue"].sum())

In [11]:
overall_kpis = pd.DataFrame({
    "metric": [
        "Gross revenue",
        "Completed orders",
        "Units sold",
        "Unique known customers",
        "Average order value",
        "Average items per order",
        "Average revenue per known customer",
        "Cancelled invoices",
        "Cancellation rate (%)",
        "Negative-quantity units",
        "Negative-quantity value",
    ],
    "value": [
        gross_revenue,
        completed_orders,
        units_sold,
        unique_customers,
        average_order_value,
        average_items_per_order,
        average_revenue_per_customer,
        cancelled_invoice_count,
        cancellation_rate,
        returned_or_cancelled_units,
        negative_quantity_value,
    ],
})

overall_kpis

,metric,value
0,Gross revenue,1.064211e+07
1,Completed orders,1.996000e+04
2,Units sold,5.572420e+06
3,Unique known customers,4.338000e+03
4,Average order value,5.331719e+02
5,Average items per order,2.791794e+02
6,Average revenue per known customer,2.048688e+03
7,Cancelled invoices,3.836000e+03
8,Cancellation rate (%),1.569173e+01
9,Negative-quantity units,2.755600e+05


In [12]:
overall_kpis_display = overall_kpis.copy()

overall_kpis_display["value"] = (overall_kpis_display["value"].round(2))

overall_kpis_display

,metric,value
0,Gross revenue,10642110.80
1,Completed orders,19960.00
2,Units sold,5572420.00
3,Unique known customers,4338.00
4,Average order value,533.17
5,Average items per order,279.18
6,Average revenue per known customer,2048.69
7,Cancelled invoices,3836.00
8,Cancellation rate (%),15.69
9,Negative-quantity units,275560.00


## 5. Order-Level Analysis

In [13]:
order_kpis = (
    sales_df.groupby("invoice_no",as_index=False,)
    .agg(
        order_revenue=("line_revenue","sum"),
        units=("quantity","sum"),
        product_lines=("stock_code","size"),
        unique_products=("stock_code","nunique"),
        customer_id=("customer_id","first"),
        country=("country","first"),
        order_date=("invoice_date","min"),
    )
)

In [14]:
order_kpis.head()

,invoice_no,order_revenue,units,product_lines,unique_products,customer_id,country,order_date
0,536365,139.12,40,7,7,17850,United Kingdom,2010-12-01 08:26:00
1,536366,22.20,12,2,2,17850,United Kingdom,2010-12-01 08:28:00
2,536367,278.73,83,12,12,13047,United Kingdom,2010-12-01 08:34:00
3,536368,70.05,15,4,4,13047,United Kingdom,2010-12-01 08:34:00
4,536369,17.85,3,1,1,13047,United Kingdom,2010-12-01 08:35:00


In [15]:
assert len(order_kpis) == completed_orders
assert (order_kpis["order_revenue"] > 0).all()

print(f"Order rows: {len(order_kpis):,}")

Order rows: 19,960


In [16]:
order_kpis[["order_revenue","units","unique_products"]].describe()

,order_revenue,units,unique_products
count,19960.000000,19960.000000,19960.000000
mean,533.171884,279.179359,26.032164
std,1780.412288,955.011810,46.982195
min,0.380000,1.000000,1.000000
25%,151.695000,69.000000,6.000000
50%,303.300000,150.000000,15.000000
75%,493.462500,296.000000,29.000000
max,168469.600000,80995.000000,1110.000000


In [17]:
order_kpis.nlargest(10,"order_revenue")

,invoice_no,order_revenue,units,product_lines,unique_products,customer_id,country,order_date
19923,581483,168469.60,80995,1,1,16446,United Kingdom,2011-12-09 09:15:00
2116,541431,77183.60,74215,1,1,12346,United Kingdom,2011-01-18 10:01:00
16923,574941,52940.94,14149,101,101,<NA>,United Kingdom,2011-11-07 17:42:00
17579,576365,50653.91,13956,99,99,<NA>,United Kingdom,2011-11-14 17:55:00
8693,556444,38970.00,60,1,1,15098,United Kingdom,2011-06-10 15:28:00
13545,567423,31698.16,12572,12,12,17450,United Kingdom,2011-09-20 11:05:00
8899,556917,22775.93,15049,138,138,12415,Australia,2011-06-15 13:37:00
15693,572209,22206.00,1920,7,7,18102,United Kingdom,2011-10-21 12:08:00
13537,567381,22104.80,6760,12,12,17450,United Kingdom,2011-09-20 10:12:00
11881,563614,21880.44,12196,97,97,12415,Australia,2011-08-18 08:51:00


## 6. Monthly Performance

In [18]:
monthly_kpis = (sales_df.groupby("invoice_month",as_index=False)
    .agg(
        revenue=("line_revenue","sum"),
        orders=("invoice_no","nunique"),
        customers=("customer_id","nunique"),
        units_sold=("quantity","sum"),
    )
)

In [19]:
monthly_kpis["average_order_value"] = (monthly_kpis["revenue"]/ monthly_kpis["orders"])
monthly_kpis["items_per_order"] = (monthly_kpis["units_sold"]/ monthly_kpis["orders"])

In [20]:
monthly_kpis["revenue_growth_pct"] = (monthly_kpis["revenue"].pct_change()*100)

In [21]:
monthly_kpis = monthly_kpis.sort_values("invoice_month").reset_index(drop=True)

In [22]:
monthly_kpis.round(2)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_14380\563567013.py:1: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  monthly_kpis.round(2)


,invoice_month,revenue,orders,customers,units_sold,average_order_value,items_per_order,revenue_growth_pct
0,2010-12-01,821452.73,1559,885,358019,526.91,229.65,NaN
1,2011-01-01,689811.61,1086,741,387099,635.19,356.44,-16.03
2,2011-02-01,522545.56,1100,758,282934,475.04,257.21,-24.25
3,2011-03-01,716215.26,1454,974,376599,492.58,259.01,37.06
4,2011-04-01,536968.49,1246,856,307953,430.95,247.15,-25.03
5,2011-05-01,769296.61,1681,1056,395001,457.64,234.98,43.27
6,2011-06-01,760547.01,1533,991,388511,496.12,253.43,-1.14
7,2011-07-01,718076.12,1475,949,399693,486.83,270.98,-5.58
8,2011-08-01,757841.38,1361,935,421020,556.83,309.35,5.54
9,2011-09-01,1056435.19,1837,1266,569573,575.09,310.06,39.40


In [23]:
best_month = monthly_kpis.loc[monthly_kpis["revenue"].idxmax()]

best_month

invoice_month          2011-11-01 00:00:00
revenue                         1503866.78
orders                                2769
customers                             1664
units_sold                          751377
average_order_value             543.108263
items_per_order                 271.353196
revenue_growth_pct               30.627478
Name: 11, dtype: object

In [24]:
lowest_month = monthly_kpis.loc[monthly_kpis["revenue"].idxmin()]

lowest_month

invoice_month          2011-02-01 00:00:00
revenue                          522545.56
orders                                1100
customers                              758
units_sold                          282934
average_order_value             475.041418
items_per_order                 257.212727
revenue_growth_pct              -24.248077
Name: 2, dtype: object

In [25]:
print("Highest revenue month:",best_month["invoice_month"].strftime("%Y-%m"))

print(f"Revenue: £{best_month['revenue']:,.2f}")

Highest revenue month: 2011-11
Revenue: £1,503,866.78


In [26]:
print("First transaction:",sales_df["invoice_date"].min())

print("Last transaction:",sales_df["invoice_date"].max())

First transaction: 2010-12-01 08:26:00
Last transaction: 2011-12-09 12:50:00


## 7. Country Performance

In [27]:
country_kpis = (sales_df.groupby("country",as_index=False)
    .agg(
        revenue=("line_revenue","sum"),
        orders=("invoice_no","nunique"),
        customers=("customer_id","nunique"),
        units_sold=("quantity","sum"),
    )
)

In [28]:
country_order_metrics = (
    order_kpis.groupby("country",as_index=False)
    .agg(
        average_order_value=("order_revenue","mean"),
        median_order_value=("order_revenue","median"),
    )
)

In [29]:
country_kpis = country_kpis.merge(country_order_metrics,on="country",how="left",)

In [30]:
country_kpis["revenue_share_pct"] = (country_kpis["revenue"]/ country_kpis["revenue"].sum()* 100)

In [31]:
country_kpis = (country_kpis.sort_values("revenue",ascending=False,).reset_index(drop=True))

In [32]:
country_kpis["revenue_rank"] = (country_kpis["revenue"].rank(method="dense",ascending=False,).astype(int))

In [33]:
country_kpis.head(10).round(2)

,country,revenue,orders,customers,units_sold,average_order_value,median_order_value,revenue_share_pct,revenue_rank
0,United Kingdom,9001744.09,18019,3920,4646906,499.57,299.95,84.59,1
1,Netherlands,285446.34,94,9,200361,3036.66,806.88,2.68,2
2,EIRE,283140.52,288,3,147007,983.13,651.72,2.66,3
3,Germany,228678.40,457,94,119154,500.39,354.85,2.15,4
4,France,209625.37,392,87,112060,534.76,375.14,1.97,5
5,Australia,138453.81,57,9,83891,2429.01,429.60,1.30,6
6,Spain,61558.56,90,30,27933,683.98,431.89,0.58,7
7,Switzerland,57067.60,54,21,30617,1056.81,625.96,0.54,8
8,Belgium,41196.34,98,25,23237,420.37,346.39,0.39,9
9,Sweden,38367.83,36,8,36078,1065.77,366.96,0.36,10


In [34]:
international_country_kpis = (country_kpis[country_kpis["country"]!= "United Kingdom"].copy())

Outside of the UK

In [35]:
international_country_kpis.head(10).round(2)

,country,revenue,orders,customers,units_sold,average_order_value,median_order_value,revenue_share_pct,revenue_rank
1,Netherlands,285446.34,94,9,200361,3036.66,806.88,2.68,2
2,EIRE,283140.52,288,3,147007,983.13,651.72,2.66,3
3,Germany,228678.40,457,94,119154,500.39,354.85,2.15,4
4,France,209625.37,392,87,112060,534.76,375.14,1.97,5
5,Australia,138453.81,57,9,83891,2429.01,429.60,1.30,6
6,Spain,61558.56,90,30,27933,683.98,431.89,0.58,7
7,Switzerland,57067.60,54,21,30617,1056.81,625.96,0.54,8
8,Belgium,41196.34,98,25,23237,420.37,346.39,0.39,9
9,Sweden,38367.83,36,8,36078,1065.77,366.96,0.36,10
10,Japan,37416.37,19,8,26016,1969.28,1607.04,0.35,11


## 8. Product Performance

In [36]:
product_kpis = (sales_df.groupby(["stock_code","description",],as_index=False,)
    .agg(
        revenue = ("line_revenue","sum"),
        units_sold = ("quantity","sum"),
        orders = ("invoice_no","nunique"),
        customers = ("customer_id","nunique")
    )
)

In [37]:
product_kpis["average_selling_price"] = (product_kpis["revenue"]/ product_kpis["units_sold"])

In [38]:
product_kpis["revenue_share_pct"] = (product_kpis["revenue"]/ product_kpis["revenue"].sum()* 100)

In [39]:
product_kpis["revenue_rank"] = (product_kpis["revenue"].rank(method="dense",ascending=False,).astype(int))

In [40]:
product_kpis = (product_kpis.sort_values("revenue",ascending=False,).reset_index(drop=True))

In [41]:
product_kpis.head(10).round(2)

,stock_code,description,revenue,units_sold,orders,customers,average_selling_price,revenue_share_pct,revenue_rank
0,DOT,DOTCOM POSTAGE,206248.77,706,706,1,292.14,1.94,1
1,22423,REGENCY CAKESTAND 3 TIER,174156.54,13851,1988,881,12.57,1.64,2
2,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1,1,2.08,1.58,3
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,104284.24,37580,2189,856,2.77,0.98,4
4,47566,PARTY BUNTING,99445.23,18283,1685,708,5.44,0.93,5
5,85099B,JUMBO BAG RED RETROSPOT,94159.81,48371,2089,635,1.95,0.88,6
6,23166,MEDIUM CERAMIC TOP STORAGE JAR,81700.92,78033,247,138,1.05,0.77,7
7,POST,POSTAGE,78101.88,3150,1126,331,24.79,0.73,8
8,M,Manual,77750.27,6984,289,197,11.13,0.73,9
9,23084,RABBIT NIGHT LIGHT,66870.03,30739,994,450,2.18,0.63,10


In [42]:
product_kpis.nlargest(10,"units_sold",).round(2)

,stock_code,description,revenue,units_sold,orders,customers,average_selling_price,revenue_share_pct,revenue_rank
2,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1,1,2.08,1.58,3
6,23166,MEDIUM CERAMIC TOP STORAGE JAR,81700.92,78033,247,138,1.05,0.77,7
146,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,13814.01,54951,535,307,0.25,0.13,147
5,85099B,JUMBO BAG RED RETROSPOT,94159.81,48371,2089,635,1.95,0.88,6
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,104284.24,37580,2189,856,2.77,0.98,4
22,22197,POPCORN HOLDER,34288.67,36749,803,295,0.93,0.32,23
84,21212,PACK OF 72 RETROSPOT CAKE CASES,21246.45,36396,1320,635,0.58,0.20,85
11,84879,ASSORTED COLOUR BIRD ORNAMENT,58927.62,36362,1455,678,1.62,0.55,12
9,23084,RABBIT NIGHT LIGHT,66870.03,30739,994,450,2.18,0.63,10
115,22492,MINI PAINT SET VINTAGE,16937.82,26633,380,213,0.64,0.16,116


## 9. Customer Performance

In [43]:
customer_kpis = (customer_sales_df.groupby("customer_id",as_index=False,)
    .agg(
        revenue=("line_revenue", "sum"),
        orders=("invoice_no", "nunique"),
        units_purchased=("quantity", "sum"),
        unique_products=("stock_code","nunique"),
        first_purchase=("invoice_date","min"),
        last_purchase=("invoice_date", "max"),
        country=("country","first")
    )
)

In [44]:
customer_kpis["average_order_value"] = (customer_kpis["revenue"]/ customer_kpis["orders"])

In [45]:
customer_kpis["revenue_rank"] = (customer_kpis["revenue"].rank(method="dense",ascending=False,).astype(int))

In [46]:
customer_kpis = (customer_kpis.sort_values("revenue",ascending=False,).reset_index(drop=True))

In [47]:
customer_kpis.head(10).round(2)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_14380\3683168288.py:1: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  customer_kpis.head(10).round(2)


,customer_id,revenue,orders,units_purchased,unique_products,first_purchase,last_purchase,country,average_order_value,revenue_rank
0,14646,280206.02,73,196915,700,2010-12-20 10:09:00,2011-12-08 12:12:00,Netherlands,3838.44,1
1,18102,259657.30,60,64124,150,2010-12-07 16:42:00,2011-12-09 11:50:00,United Kingdom,4327.62,2
2,17450,194390.79,46,69973,124,2010-12-07 09:23:00,2011-12-01 13:29:00,United Kingdom,4225.89,3
3,16446,168472.50,2,80997,3,2011-05-18 09:52:00,2011-12-09 09:15:00,United Kingdom,84236.25,4
4,14911,143711.17,201,80240,1787,2010-12-01 14:05:00,2011-12-08 15:54:00,EIRE,714.98,5
5,12415,124914.53,21,77374,444,2011-01-06 11:12:00,2011-11-15 14:22:00,Australia,5948.31,6
6,14156,117210.08,55,57768,714,2010-12-03 11:48:00,2011-11-30 10:54:00,EIRE,2131.09,7
7,17511,91062.38,31,64549,453,2010-12-01 10:19:00,2011-12-07 10:12:00,United Kingdom,2937.50,8
8,16029,80850.84,63,40108,44,2010-12-01 09:57:00,2011-11-01 10:27:00,United Kingdom,1283.35,9
9,12346,77183.60,1,74215,1,2011-01-18 10:01:00,2011-01-18 10:01:00,United Kingdom,77183.60,10


## 10. Cancellation and Return Activity

In [48]:
transaction_type_summary = (transactions_df.groupby("transaction_type",as_index=False,)
    .agg(
        rows=("invoice_no","size"),
        invoices=("invoice_no",  "nunique"),
        units=("quantity","sum"),
        recorded_value=("line_revenue","sum")
    )
)

In [49]:
transaction_type_summary["row_share_pct"] = (transaction_type_summary["rows"]/ transaction_type_summary["rows"].sum()* 100)

In [50]:
transaction_type_summary.round(2)

,transaction_type,rows,invoices,units,recorded_value,row_share_pct
0,cancelled,9251,3836,-275560,-893979.73,1.73
1,invalid_price,584,229,40052,-22124.12,0.11
2,return,474,474,-160801,0.00,0.09
3,sale,524878,19960,5572420,10642110.80,98.07


## 11. Revenue Concentration

In [51]:
top_10_product_revenue = (product_kpis.head(10)["revenue"].sum())

top_10_product_share = (top_10_product_revenue/ product_kpis["revenue"].sum()* 100)

In [52]:
top_10_customer_revenue = (customer_kpis.head(10)["revenue"].sum())

top_10_customer_share = (top_10_customer_revenue/ customer_kpis["revenue"].sum()* 100)

In [53]:
print("Top 10 product revenue share:",f"{top_10_product_share:.2f}%",)

print( "Top 10 customer revenue share:",f"{top_10_customer_share:.2f}%",)

Top 10 product revenue share: 10.82%
Top 10 customer revenue share: 17.30%


In [54]:
product_kpis["cumulative_revenue_pct"] = (product_kpis["revenue"].cumsum()/ product_kpis["revenue"].sum()* 100)

In [55]:
products_for_80_pct = (product_kpis["cumulative_revenue_pct"].searchsorted(80)+ 1)

print("Products required to reach 80% "f"of revenue: {products_for_80_pct:,}")

Products required to reach 80% of revenue: 836


In [56]:
top_five_countries = (country_kpis.head(5)["country"].tolist())

In [57]:
top_country_sales = sales_df[sales_df["country"].isin(top_five_countries)].copy()

In [58]:
monthly_country_pivot = (top_country_sales.pivot_table(
        index="invoice_month",
        columns="country",
        values="line_revenue",
        aggfunc="sum",
        fill_value=0,
    )
)

In [59]:
monthly_country_pivot.round(2)

country,EIRE,France,Germany,Netherlands,United Kingdom
invoice_month,,,,,
2010-12-01,10033.26,9616.31,15205.74,8784.48,746082.22
2011-01-01,21904.19,17740.12,16880.84,26611.16,559975.83
2011-02-01,12203.74,8515.96,9581.05,23011.91,428986.62
2011-03-01,22197.51,14584.95,14392.69,22416.49,584810.78
2011-04-01,7570.50,5529.61,12315.54,2976.56,475677.63
2011-05-01,18003.72,17606.48,25734.70,29185.88,638104.89
2011-06-01,24863.61,16348.41,13246.35,26858.09,618345.53
2011-07-01,42738.85,10000.19,16440.98,26.02,600919.07
2011-08-01,18344.20,13810.96,19220.77,40327.81,606784.93


## 12. Key Findings

In [60]:
top_country = country_kpis.iloc[0]
top_product = product_kpis.iloc[0]
top_customer = customer_kpis.iloc[0]
best_month = monthly_kpis.loc[
    monthly_kpis["revenue"].idxmax()
]

In [61]:
print(
    f"Gross revenue was "
    f"£{gross_revenue:,.2f} from "
    f"{completed_orders:,} completed orders."
)

print(
    f"Average order value was "
    f"£{average_order_value:,.2f}."
)

print(
    f"The highest-revenue country was "
    f"{top_country['country']}, generating "
    f"£{top_country['revenue']:,.2f}."
)

print(
    f"The highest-revenue product was "
    f"{top_product['description']}, generating "
    f"£{top_product['revenue']:,.2f}."
)

print(
    f"The strongest recorded month was "
    f"{best_month['invoice_month']:%Y-%m}, "
    f"with £{best_month['revenue']:,.2f} "
    f"in revenue."
)

print(
    f"The top 10 products generated "
    f"{top_10_product_share:.2f}% "
    f"of total gross revenue."
)

print(
    f"The top 10 known customers generated "
    f"{top_10_customer_share:.2f}% "
    f"of known-customer revenue."
)

Gross revenue was £10,642,110.80 from 19,960 completed orders.
Average order value was £533.17.
The highest-revenue country was United Kingdom, generating £9,001,744.09.
The highest-revenue product was DOTCOM POSTAGE, generating £206,248.77.
The strongest recorded month was 2011-11, with £1,503,866.78 in revenue.
The top 10 products generated 10.82% of total gross revenue.
The top 10 known customers generated 17.30% of known-customer revenue.


## Initial Business Interpretation

1. **Overall Performance:**
The business shows strong performance! The dataset records a total revenue of £10,642,110.80 
generated from 19,962 completed orders, resulting in a solid Average Order Value (AOV) of 
£543.11.

2. **Monthly Performance:**
November 2011 stands out as the peak month for sales. However, the data for the very first and 
last months appears to be incomplete. These periods should be excluded before comparing monthly 
trends to avoid skewed results.

3. **Market Concentration:**
The United Kingdom drives the vast majority of the sales. This indicates a high dependence on a 
single market, exposing the business to potential regional risks.

4. **Product Performance:**
Interestingly, the top revenue generator isn't a physical item, but DOTCOM POSTAGE (shipping fees). 
This finding also highlights that products with the highest sales volume do not always bring in the 
most money for the company.

5. **Customer Concentration:**
The Top 10 VIP customers account for 10.82% of the known customer revenue. This reflects a moderate 
dependence on a small buyer group, suggesting that the company should maintain a targeted retention 
strategy to keep these key clients engaged.

6. **Cancellations:**
The overall cancellation rate is notably low at just 1.73%. As a next analytical step, it would be 
highly beneficial to drill down and see if these canceled invoices are tied to specific products or 
countries to identify any root causes.

## 13. Export Analysis Tables

In [62]:
output_paths = {
    "overall_kpis": (reports_table_dir/ "overall_kpis.csv"),
    "monthly_kpis": (reports_table_dir/ "monthly_kpis.csv"),
    "country_kpis": (reports_table_dir/ "country_kpis.csv"),
    "product_kpis": (reports_table_dir/ "product_kpis.csv"),
    "customer_kpis": (reports_table_dir/ "customer_kpis.csv"),
    "order_kpis": (reports_table_dir/ "order_kpis.csv"),
    "transaction_type_summary": (reports_table_dir/ "transaction_type_summary.csv"),
    "monthly_country_pivot": (reports_table_dir/ "monthly_country_pivot.csv"),
}

In [63]:
overall_kpis.to_csv(output_paths["overall_kpis"],index=False,)

monthly_kpis.to_csv(output_paths["monthly_kpis"],index=False,)

country_kpis.to_csv(output_paths["country_kpis"],index=False,)

product_kpis.to_csv(output_paths["product_kpis"],index=False,)

customer_kpis.to_csv(output_paths["customer_kpis"],index=False,)

order_kpis.to_csv(output_paths["order_kpis"],index=False,)

transaction_type_summary.to_csv(output_paths["transaction_type_summary"],index=False,)

monthly_country_pivot.to_csv(output_paths["monthly_country_pivot"],)

In [64]:
for name, path in output_paths.items():
    print(f"{name}: {path.exists()}")

overall_kpis: True
monthly_kpis: True
country_kpis: True
product_kpis: True
customer_kpis: True
order_kpis: True
transaction_type_summary: True
monthly_country_pivot: True


## 14. Conclusion

The cleaned sales data was converted into overall, monthly, country, product,
customer, and order-level business summaries.

The analysis calculated gross revenue, completed orders, average order value,
units sold, known customers, cancellation activity, and revenue concentration.
Separate tables were created for each area so they can be reused for
visualization and reporting.

The next stage will use these tables to create clear charts and communicate the
most important business findings.